# CSE 151B — Submission notebook (`private.jsonl`)

Live in **`notebooks/submission.ipynb`**. Same inference stack as [`full_public.ipynb`](full_public.ipynb), run on all of **`data/private.jsonl`** (no labels).

1. (**Colab A100**) `%pip` → restart runtime → Drive copy cell.
2. Set **`MAX_TOKENS`** in §2 (default **32k**, matching pub-003).
3. Prompts are **per question**: baseline for MCQ and single-blank free-form; multi-blank format when `[ANS]` count > 1.
4. Generate with checkpointing → write **`results/submission.csv`**.

**Output:** CSV with columns `id`, `response` (full model trace, CSV-escaped). Checkpoint: `results/submission.responses.jsonl`.

### Google Colab

**Colab (A100):** run the `%pip` cell below, restart runtime, continue. **Local:** use your venv — same packages (`vllm`, `transformers`, …).

> **Note:** `bitsandbytes` is not needed — Qwen3-4B fits in native bf16 on A100 (~8 GB), which is faster than quantized loading.

In [ ]:
# Colab: skip when working locally with an existing venv.
%pip install -q sympy numpy tqdm "antlr4-python3-runtime==4.11.1"
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu129 --upgrade
%pip install -q "https://github.com/vllm-project/vllm/releases/download/v0.20.0/vllm-0.20.0+cu129-cp38-abi3-manylinux_2_31_x86_64.whl" --extra-index-url https://download.pytorch.org/whl/cu129
%pip install -q transformers

## 2. Imports & configuration

- `MAX_TOKENS` — **32768** = pub-003 path (default)
- `SUBMISSION_CSV` — `results/submission.csv`
- `DATA_PATH` — `data/private.jsonl`

Prompts are chosen automatically in §5 (no variant knob).

In [ ]:
import csv
import json
import os
import re
from collections import Counter
from fractions import Fraction
from pathlib import Path


def _repo_root() -> Path:
    here = Path.cwd().resolve()
    if (here / "data").is_dir():
        return here
    if here.name == "notebooks" and (here.parent / "data").is_dir():
        return here.parent
    up = here.parent
    if (up / "data").is_dir():
        return up
    return here


REPO_ROOT = _repo_root()

MAX_TOKENS = 32768

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID   = "0"
DATA_PATH = str(REPO_ROOT / "data/private.jsonl")
SUBMISSION_CSV = str(REPO_ROOT / "results/submission.csv")

# Self-consistency settings
SELF_CONSISTENCY_K = 6

# Only run verifier when vote agreement is weak or output format looks bad.
USE_VERIFIER = True
VERIFY_AGREEMENT_THRESHOLD = 0.67

SUBMIT_FINAL_ONLY = True

_TOKEN_K = MAX_TOKENS // 1024

print(f"REPO_ROOT      = {REPO_ROOT}")
print(f"MAX_TOKENS     = {MAX_TOKENS} ({_TOKEN_K}k)")
print(f"SELF_CONSISTENCY_K = {SELF_CONSISTENCY_K}")
print(f"USE_VERIFIER   = {USE_VERIFIER}")
print(f"SUBMIT_FINAL_ONLY = {SUBMIT_FINAL_ONLY}")
print(f"SUBMISSION_CSV = {SUBMISSION_CSV}")

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

import glob
import site

def _prepend_nvidia_libs_to_ld_path() -> None:
    roots = list(site.getsitepackages())
    user_site = getattr(site, "getusersitepackages", lambda: None)()
    if user_site:
        roots.append(user_site)
    libdirs: list[str] = []
    seen: set[str] = set()
    for root in roots:
        for d in glob.glob(os.path.join(root, "nvidia", "*", "lib")):
            if os.path.isdir(d) and d not in seen:
                seen.add(d)
                libdirs.append(d)
    if libdirs:
        sep = os.pathsep
        os.environ["LD_LIBRARY_PATH"] = sep.join(libdirs + [os.environ.get("LD_LIBRARY_PATH", "")]).strip(sep)


_prepend_nvidia_libs_to_ld_path()

import contextlib
import io
import sys
from typing import Optional

@contextlib.contextmanager
def _jupyter_stdout_for_vllm():
    try:
        sys.stdout.fileno()
    except (io.UnsupportedOperation, AttributeError, OSError):
        saved_out, saved_err = sys.stdout, sys.stderr
        dup1, dup2 = os.dup(1), os.dup(2)
        try:
            sys.stdout = os.fdopen(dup1, "w", buffering=1)
            sys.stderr = os.fdopen(dup2, "w", buffering=1)
            yield
        finally:
            sys.stdout.close()
            sys.stderr.close()
            sys.stdout, sys.stderr = saved_out, saved_err
    else:
        yield

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

## 3. Colab: copy `private.jsonl` from Drive

Upload **`private.jsonl`** to `My Drive/CSE151B/data/private.jsonl` (or change `DRIVE_PRIVATE`). Skip on local clone when `data/private.jsonl` exists.

In [ ]:
import shutil
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    print("Skip: not Google Colab — use repo `data/private.jsonl`.")
else:
    drive.mount("/content/drive")
    DRIVE_PRIVATE = Path("/content/drive/MyDrive/CSE151B/data/private.jsonl")
    DRIVE_PROJECT_ROOT = DRIVE_PRIVATE.parent.parent
    if not DRIVE_PRIVATE.is_file():
        raise FileNotFoundError(
            f"Missing on Drive: {DRIVE_PRIVATE}\nUpload private.jsonl or set DRIVE_PRIVATE."
        )
    (REPO_ROOT / "data").mkdir(parents=True, exist_ok=True)
    dest = REPO_ROOT / "data/private.jsonl"
    shutil.copy2(DRIVE_PRIVATE, dest)
    print(f"Copied to {dest.resolve()}")

## 4. Load dataset

Private rows have `question`, optional `options`, and `id` — **no** `answer` field.

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |

In [ ]:
def n_ans_placeholders(question: str) -> int:
    return question.count("[ANS]")


data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options") for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form) from {DATA_PATH}")

mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))
multi_sample = next(
    (d for d in data if not d.get("options") and n_ans_placeholders(d["question"]) > 1),
    free_sample,
)

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2)[:1200], "...\n" if len(json.dumps(mcq_sample)) > 1200 else "\n")
print("── Free-form sample ──")
print(json.dumps(free_sample, indent=2)[:1200], "...\n" if len(json.dumps(free_sample)) > 1200 else "\n")
if multi_sample is not free_sample:
    print(f"── Multi-blank sample ({n_ans_placeholders(multi_sample['question'])} blanks) ──")
    print(json.dumps(multi_sample, indent=2)[:1200], "...\n" if len(json.dumps(multi_sample)) > 1200 else "\n")

## 5. Prompt construction

Per question — no global variant switch:

| Case | System prompt |
|------|----------------|
| MCQ | baseline |
| Free-form, 1 `[ANS]` | baseline |
| Free-form, 2+ `[ANS]` | multi-blank (`\boxed{a}, \boxed{b}, ...`, judger-compatible) |

In [ ]:
_MATH_BASELINE = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Be careful with arithmetic, signs, units, and exact values. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

_MCQ_BASELINE = (
    "You are an expert mathematician. "
    "Solve the problem independently before looking for the matching answer choice. "
    "Then compare your result against the choices. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)

_MATH_MULTI_BLANK = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "For problems with multiple [ANS] placeholders, put each sub-answer in its own "
    "\\boxed{}, separated by commas, in the order the blanks appear "
    "(e.g. \\boxed{3}, \\boxed{7}). Do not use labels like 'Answer 1:' between boxes. "
    "For single-answer problems, use one \\boxed{}."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return _MCQ_BASELINE, f"{question}\n\nOptions:\n{opts_text}"
    n_blanks = n_ans_placeholders(question)
    if n_blanks > 1:
        example = ", ".join("\\boxed{...}" for _ in range(n_blanks))
        user = (
            f"{question}\n\n"
            f"The problem has {n_blanks} [ANS] blanks. "
            f"After your reasoning, give {n_blanks} comma-separated \\boxed{{}} values "
            f"in order: {example}"
        )
        return _MATH_MULTI_BLANK, user
    return _MATH_BASELINE, question


def prompt_mode(question: str, options: Optional[list]) -> str:
    if options:
        return "mcq/baseline"
    return "multi-blank" if n_ans_placeholders(question) > 1 else "baseline"


for label, item in [
    ("MCQ", mcq_sample),
    ("Free-form (single-blank)", free_sample),
    *( [(f"Multi-blank ({n_ans_placeholders(multi_sample['question'])} blanks)", multi_sample)] if multi_sample is not free_sample else [] ),
]:
    mode = prompt_mode(item["question"], item.get("options"))
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} [{mode}] ──")
    print(f"  system: {sys_p[:120]}...")
    print(f"  user (first 300 chars): {usr_p[:300]}...\n")

In [ ]:
def strip_thinking(text: str) -> str:
    """Keep only final text after </think>, if present."""
    text = text.strip()
    if "</think>" in text:
        return text.split("</think>")[-1].strip()
    return text


def extract_boxed(text: str) -> list[str]:
    """Extract balanced \\boxed{...} contents, including nested braces like \\frac{1}{2}."""
    text = strip_thinking(text)
    boxes = []
    token = "\\boxed"
    i = 0

    while True:
        j = text.find(token, i)
        if j == -1:
            break

        k = j + len(token)
        while k < len(text) and text[k].isspace():
            k += 1

        if k >= len(text) or text[k] != "{":
            i = k
            continue

        depth = 0
        start = k + 1
        end = None

        for p in range(k, len(text)):
            if text[p] == "{":
                depth += 1
            elif text[p] == "}":
                depth -= 1
                if depth == 0:
                    end = p
                    break

        if end is None:
            break

        boxes.append(text[start:end].strip())
        i = end + 1

    return boxes


def normalize_answer(ans: str) -> str:
    """Normalize equivalent-looking final answers for voting."""
    if ans is None:
        return ""

    s = str(ans).strip()
    s = s.strip("$")
    s = s.replace("−", "-")
    s = s.replace("\\left", "").replace("\\right", "")
    s = s.replace("\\,", "").replace("\\!", "")
    s = s.replace("\\dfrac", "\\frac").replace("\\tfrac", "\\frac")
    s = re.sub(r"\\text\{([^{}]*)\}", r"\1", s)
    s = re.sub(r"\s+", "", s)
    s = s.rstrip(".")

    # Normalize simple LaTeX fractions.
    m = re.fullmatch(r"\\frac\{(-?\d+)\}\{(-?\d+)\}", s)
    if m:
        num = int(m.group(1))
        den = int(m.group(2))
        if den != 0:
            return str(Fraction(num, den))

    # Normalize plain fractions and decimals.
    try:
        if re.fullmatch(r"-?\d+/\d+", s):
            return str(Fraction(s))
        if re.fullmatch(r"-?\d+(\.\d+)?", s):
            return str(Fraction(s))
    except Exception:
        pass

    return s.lower()


def format_question_with_options(item: dict) -> str:
    question = item["question"]
    options = item.get("options")

    if not options:
        return question

    labels = [chr(65 + i) for i in range(len(options))]
    opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
    return f"{question}\n\nOptions:\n{opts_text}"


def extract_candidate(item: dict, response: str) -> tuple[tuple, str]:
    """
    Returns:
      key: normalized key used for voting
      final_response: clean boxed answer for CSV
    """
    boxes = extract_boxed(response)
    options = item.get("options")

    # MCQ handling
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        candidate = boxes[-1].strip() if boxes else strip_thinking(response).strip()
        candidate_clean = candidate.upper().strip().replace(".", "")

        if candidate_clean in labels:
            return ("mcq", candidate_clean), f"\\boxed{{{candidate_clean}}}"

        # If the model outputs the choice text instead of the letter, map it back.
        norm_candidate = normalize_answer(candidate)
        for label, opt in zip(labels, options):
            if norm_candidate == normalize_answer(opt):
                return ("mcq", label), f"\\boxed{{{label}}}"

        return ("invalid_mcq", normalize_answer(candidate)), f"\\boxed{{{candidate}}}"

    # Free-response handling
    n_blanks = n_ans_placeholders(item["question"])

    if n_blanks > 1:
        if len(boxes) >= n_blanks:
            vals = boxes[-n_blanks:]
        elif len(boxes) == 1 and "," in boxes[-1]:
            vals = [x.strip() for x in boxes[-1].split(",")]
        else:
            vals = boxes if boxes else [strip_thinking(response).strip()]

        key = tuple(normalize_answer(v) for v in vals)
        final_response = ", ".join(f"\\boxed{{{v}}}" for v in vals)
        return ("multi", key), final_response

    # Single free-response
    if boxes:
        val = boxes[-1]
    else:
        lines = [x.strip() for x in strip_thinking(response).splitlines() if x.strip()]
        val = lines[-1] if lines else strip_thinking(response).strip()

    return ("free", normalize_answer(val)), f"\\boxed{{{val}}}"


def vote_responses(item: dict, raw_responses: list[str]) -> dict:
    records = []

    for raw in raw_responses:
        key, final_response = extract_candidate(item, raw)
        records.append({
            "key": key,
            "final_response": final_response,
            "raw_response": raw,
        })

    counts = Counter(r["key"] for r in records)
    best_key, best_count = counts.most_common(1)[0]

    # Pick the first raw response that produced the winning normalized answer.
    best_record = next(r for r in records if r["key"] == best_key)

    return {
        "key": best_key,
        "agreement": best_count / len(raw_responses),
        "final_response": best_record["final_response"],
        "raw_response": best_record["raw_response"],
        "all_keys": [r["key"] for r in records],
    }


def final_format_bad(item: dict, final_response: str) -> bool:
    boxes = extract_boxed(final_response)
    options = item.get("options")

    if options:
        if len(boxes) != 1:
            return True
        labels = [chr(65 + i) for i in range(len(options))]
        return boxes[0].strip().upper() not in labels

    n_blanks = n_ans_placeholders(item["question"])
    if n_blanks > 1:
        return len(boxes) != n_blanks

    return len(boxes) != 1


def should_verify(item: dict, vote: dict) -> bool:
    if not USE_VERIFIER:
        return False

    if vote["agreement"] < VERIFY_AGREEMENT_THRESHOLD:
        return True

    if str(vote["key"][0]).startswith("invalid"):
        return True

    if final_format_bad(item, vote["final_response"]):
        return True

    return False


def build_verifier_prompt(item: dict, proposed_answer: str) -> tuple[str, str]:
    system = (
        "You are a strict math verifier. Check the proposed answer carefully. "
        "If it is correct, repeat it. If it is wrong, correct it. "
        "Return only the final answer in the required boxed format."
    )

    n_blanks = n_ans_placeholders(item["question"])
    if item.get("options"):
        required = "Return exactly one option letter inside \\boxed{}, e.g. \\boxed{C}."
    elif n_blanks > 1:
        example = ", ".join("\\boxed{...}" for _ in range(n_blanks))
        required = f"Return exactly {n_blanks} boxed answers in order: {example}."
    else:
        required = "Return exactly one answer inside \\boxed{}."

    user = (
        f"Problem:\n{format_question_with_options(item)}\n\n"
        f"Proposed answer:\n{proposed_answer}\n\n"
        f"{required}"
    )

    return system, user

## 6. Load model with vLLM (A100 profile)

Same **bf16** profile as `full_public.ipynb` — not the starter L4 INT8 path. See [`docs/infra/vllm-inference-config.md`](../docs/infra/vllm-inference-config.md).

| Parameter | Value |
|-----------|-------|
| `dtype` | `bfloat16` |
| `max_model_len` | 32768 |
| `enable_prefix_caching` | True |
| `enable_chunked_prefill` | True |

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

with _jupyter_stdout_for_vllm():
    llm = LLM(
        model=MODEL_ID,
        dtype="bfloat16",
        enable_prefix_caching=True,
        gpu_memory_utilization=0.92,
        max_model_len=32768,
        trust_remote_code=True,
        max_num_seqs=128,
        max_num_batched_tokens=32768,
        enable_chunked_prefill=True,
    )

answer_sampling_params = SamplingParams(
    n=SELF_CONSISTENCY_K,
    max_tokens=MAX_TOKENS,
    temperature=0.7,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
    seed=None,
)

verifier_sampling_params = SamplingParams(
    n=1,
    max_tokens=4096,
    temperature=0.0,
    top_p=1.0,
    top_k=-1,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
    seed=123,
)

print("Model loaded.")

## 7. Generate responses

Checkpoint: `results/submission.responses.jsonl` (Drive on Colab). Delete checkpoint to regenerate from scratch.

In [ ]:
# Lower chunk size because each prompt now creates SELF_CONSISTENCY_K outputs.
# On A100, 16 is a safe starting point. Try 24 or 32 if memory is stable.
CHUNK_SIZE = max(8, 96 // SELF_CONSISTENCY_K)

try:
    _results_dir = DRIVE_PROJECT_ROOT / "results"
except NameError:
    _results_dir = Path(SUBMISSION_CSV).parent

_results_dir.mkdir(parents=True, exist_ok=True)

# New checkpoint name so old single-sample checkpoints do not get mixed with voted results.
response_checkpoint = _results_dir / (
    Path(SUBMISSION_CSV).stem
    + f".k{SELF_CONSISTENCY_K}.voted.responses.jsonl"
)

print(f"Checkpoint path: {response_checkpoint}")
print(f"CHUNK_SIZE: {CHUNK_SIZE}")

completed_records: dict = {}

if response_checkpoint.exists():
    with open(response_checkpoint, encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            completed_records[r["id"]] = r
    print(f"Resumed from checkpoint: {len(completed_records)} responses already generated")

remaining = [d for d in data if d.get("id") not in completed_records]
print(f"Generating {len(remaining)} remaining / {len(data)} total...")

for chunk_start in range(0, len(remaining), CHUNK_SIZE):
    chunk = remaining[chunk_start : chunk_start + CHUNK_SIZE]

    prompts = []
    for item in chunk:
        system, user = build_prompt(item["question"], item.get("options"))
        prompt_text = tokenizer.apply_chat_template(
            [
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
            tokenize=False,
            add_generation_prompt=True,
        )
        prompts.append(prompt_text)

    with _jupyter_stdout_for_vllm():
        outputs = llm.generate(prompts, sampling_params=answer_sampling_params)

    records = []
    verifier_prompts = []
    verifier_indices = []

    for idx, (item, out) in enumerate(zip(chunk, outputs)):
        raw_samples = [sample.text.strip() for sample in out.outputs]
        vote = vote_responses(item, raw_samples)

        response_for_csv = vote["final_response"] if SUBMIT_FINAL_ONLY else vote["raw_response"]

        record = {
            "id": item["id"],
            "response": response_for_csv,
            "voted_final": vote["final_response"],
            "best_raw_response": vote["raw_response"],
            "agreement": vote["agreement"],
            "vote_key": repr(vote["key"]),
            "all_vote_keys": [repr(k) for k in vote["all_keys"]],
            "raw_samples": raw_samples,
            "verified": False,
            "verifier_raw": None,
        }

        if should_verify(item, vote):
            vsystem, vuser = build_verifier_prompt(item, vote["final_response"])
            verifier_prompt = tokenizer.apply_chat_template(
                [
                    {"role": "system", "content": vsystem},
                    {"role": "user", "content": vuser},
                ],
                tokenize=False,
                add_generation_prompt=True,
            )
            verifier_prompts.append(verifier_prompt)
            verifier_indices.append(idx)

        records.append(record)

    # Verify only uncertain or badly formatted cases.
    if verifier_prompts:
        with _jupyter_stdout_for_vllm():
            verifier_outputs = llm.generate(
                verifier_prompts,
                sampling_params=verifier_sampling_params,
            )

        for record_idx, verifier_out in zip(verifier_indices, verifier_outputs):
            item = chunk[record_idx]
            verifier_raw = verifier_out.outputs[0].text.strip()
            _, verifier_final = extract_candidate(item, verifier_raw)

            records[record_idx]["verified"] = True
            records[record_idx]["verifier_raw"] = verifier_raw
            records[record_idx]["voted_final"] = verifier_final
            records[record_idx]["response"] = (
                verifier_final if SUBMIT_FINAL_ONLY else verifier_raw
            )

    with open(response_checkpoint, "a", encoding="utf-8") as f:
        for record in records:
            completed_records[record["id"]] = record
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    done = len(completed_records)
    n_verified = sum(1 for r in records if r["verified"])
    avg_agreement = sum(r["agreement"] for r in records) / len(records)

    print(
        f"  {done}/{len(data)}  "
        f"({done / len(data) * 100:.1f}%)  "
        f"verified this chunk: {n_verified}/{len(records)}  "
        f"avg agreement: {avg_agreement:.2f}"
    )

assert len(completed_records) == len(data), "Missing ids — checkpoint vs DATA_PATH mismatch"

responses = [completed_records[d["id"]]["response"] for d in data]
print(f"Done. {len(responses)} responses ready.")

## 8. Write submission CSV

Uses Python’s `csv` writer so commas and newlines inside `response` are quoted per RFC 4180.

In [ ]:
try:
    csv_out = DRIVE_PROJECT_ROOT / "results" / Path(SUBMISSION_CSV).name
except NameError:
    csv_out = Path(SUBMISSION_CSV)

csv_out.parent.mkdir(parents=True, exist_ok=True)

with open(csv_out, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f, quoting=csv.QUOTE_MINIMAL)
    w.writerow(["id", "response"])
    for row in data:
        qid = row["id"]
        w.writerow([qid, completed_records[qid]["response"]])

print(f"Wrote {len(data)} rows to {csv_out.resolve()}")

with open(csv_out, newline="", encoding="utf-8") as f:
    n = sum(1 for _ in f)
assert n == len(data) + 1, f"Expected header + {len(data)} rows, got {n} lines"
print("CSV line count OK (1 header +", len(data), "data rows).")

## Done

Upload **`submission.csv`** to the course leaderboard workflow. Keep **`submission.responses.jsonl`** as a backup of raw traces.

Pipeline matches **pub-003** (`full_public.ipynb`): 32k tokens, bf16 A100, adaptive multi-blank prompts.

In [ ]:
try:
    from google.colab import runtime
    drive.flush_and_unmount()
    runtime.unassign()
except ImportError:
    print("Not running in Colab — skipping runtime termination.")
except NameError:
    print("Drive not mounted — skipping runtime termination.")

In [3]:
!ls

sample_data
